# Lab 7b: Weighing the equatorial bulge
---

## Submission

Please upload your completed notebook (renamed with your name) to Canvas as usual, along with any scanned hand-written work if applicable.

## Introduction

In the eighteenth century the shape of the Earth was argued over with pendulums and measuring chains. Newton predicted a flattened Earth; the Cassinis' arcs in France suggested an elongated one. In 1743 Clairaut showed that the two kinds of measurement are connected: if the Earth's surface is a level surface, the way gravity increases from the equator to the poles fixes the flattening. Gravity measured at the surface therefore "weighs" the equatorial bulge.

In this lab you will repeat that experiment with modern numbers. You will

1. reduce gravity observations at stations of different heights to the reference level (the free-air reduction of Lecture 1),
2. fit a normal gravity formula $\gamma(\phi)$ to them by least squares,
3. use Clairaut's theorem (Lecture 2) to turn the fitted formula into the flattening $f$, then into $J_2$ and the moment of inertia $C/Ma^2$, and
4. find out why the eighteenth-century expeditions had to go to Lapland and to Peru.

### The data

The file `gravity_stations.npz` contains 800 (synthetic) gravity stations scattered over the globe, about a quarter of them on land. For each station you have

- `lat`, `lon`: latitude and longitude (degrees),
- `h`: height of the station above sea level (m); stations at sea are at $h = 0$,
- `g_obs`: observed gravity (mGal), already corrected for tides, drift and (at sea) the Eötvös effect,
- `land`: `True` for land stations.

The data are synthetic but realistic: they were computed from the GRS80 normal gravity formula, the free-air gradient, and the real gravity anomalies of the EGM96 gravity model at each location, plus 0.05 mGal of measurement noise. That means we know the right answers, and you can check how close your methods get to them.

## Setup

Run the cell below to download the data and import the packages.

In [ ]:
# ![ -f gravity_stations.npz ] || curl -O -s https://raw.githubusercontent.com/uafgeoteach/GEOS631_FoG/master/labs/lab_07b/gravity_stations.npz

import numpy as np
import matplotlib.pyplot as plt

data = np.load('gravity_stations.npz')
lat, lon, h, g_obs, land = data['lat'], data['lon'], data['h'], data['g_obs'], data['land']
print(f'{lat.size} stations, {land.sum()} on land')

The constants of the Geodetic Reference System 1980 (GRS80), which you will compare your results with, and two values we need from astronomy:

In [ ]:
# GRS80 (Moritz 2000)
a = 6378137.0          # semi-major axis [m]
b = 6356752.3141       # semi-minor axis [m]
GM = 3.986005e14       # [m^3 s^-2]
omega = 7.292115e-5    # angular velocity [rad/s]
J2_GRS80 = 1.08263e-3
f_GRS80 = 1 / 298.257222101

m = omega**2 * a**2 * b / GM    # rotation parameter (Lecture 2, slide 15)

# From the precession of the spin axis (Lecture 2, slide 17)
H_prec = 3.2738e-3     # dynamical ellipticity (C - A)/C

G = 6.674e-11          # [m^3 kg^-1 s^-2]
print(f'm = {m:.6e}  (about 1/{1/m:.0f})')

## Part 1: Look at the data

## <span style="color:green">Question 1</span>


In a **new code cell** below, plot `g_obs` against latitude, with land and sea stations in different colors (or color the points by height `h`). Label the axes with units.

In a **new text cell** below that, describe what you see. What is the overall trend, and how large is the change from the equator to the poles? Why is there so much more scatter among the land stations than among the sea stations?

## Part 2: Reduce to sea level

To compare stations we must refer them all to the same level. In Lecture 1 you met the free-air correction, which moves the observation from the station down to sea level as if the space in between were empty:

$$g_0 = g_\mathrm{obs} + 0.3086\,h \qquad (g \text{ in mGal}, h \text{ in m}).$$

We do **not** apply a Bouguer correction here. We are interested in what the whole Earth does, and the rock under a station is part of the Earth.

## <span style="color:green">Question 2</span>


Compute `g0`, the free-air reduced gravity for each station, and plot it against latitude as in Question 1. How large is the scatter now (you will compute the standard deviation of `g0` minus a smooth curve in Question 3; for now, describe it qualitatively)? Is the trend clearer? Answer in a **new code cell** and a **new text cell** below.

## Part 3: Fit a normal gravity formula

Clairaut's theorem (Lecture 2) says that on a level ellipsoid gravity varies with latitude as

$$\gamma(\phi) = \gamma_e\left(1 + \beta\sin^2\phi\right), \tag{1}$$

where $\gamma_e$ is gravity at the equator and $\beta = (\gamma_p - \gamma_e)/\gamma_e$ is the fractional increase toward the pole. Equation 1 is linear in the two unknowns $\gamma_e$ and $\gamma_e\beta$:

$$g_{0,i} = \gamma_e + (\gamma_e\beta)\,\sin^2\phi_i + \varepsilon_i ,$$

for each station $i$ with some noise $\varepsilon_i$. So we can estimate the two unknowns by least squares, using all of our observations ($g_{0,i}$) and a properly set up design matrix $\mathbf G$; exactly as in the network adjustment of Lecture 1: $\mathbf d = \mathbf G\mathbf m + \boldsymbol\varepsilon$ with $\mathbf d$ the vector of $g_{0,i}$, $\mathbf m = (\gamma_e, \gamma_e\beta)^\mathsf{T}$, and the design matrix

$$\mathbf G = \begin{pmatrix} 1 & \sin^2\phi_1\\ 1 & \sin^2\phi_2 \\ \vdots & \vdots \end{pmatrix}.$$

The least-squares solution is $\hat{\mathbf m} = (\mathbf G^\mathsf{T}\mathbf G)^{-1}\mathbf G^\mathsf{T}\mathbf d$ (or use `np.linalg.lstsq`), and if the misfit $\boldsymbol\varepsilon$ has variance $\sigma^2$, the covariance of the estimate is $\sigma^2(\mathbf G^\mathsf{T}\mathbf G)^{-1}$.

## <span style="color:green">Question 3</span>


(a) Build the design matrix $\mathbf G$ and estimate $\gamma_e$ and $\beta$. Estimate $\sigma^2$ from the residuals ($\sigma^2 \approx \sum r_i^2/(N - 2)$), where $N$ is the number of stations. With this, give the formal uncertainties of $\gamma_e$ and $\beta$ from calculating the covariance matrix with the formula given above.

(b) Plot the residuals $r_i = g_{0,i} - \gamma(\phi_i)$ against latitude. What is standard deviation of the residuals, and what are they physically?

(c) Compare $\gamma_e$ and $\beta$ with GRS80: $\gamma_e = 978\,032.677$ mGal, $\beta = 0.0053024$.

## Part 4: From gravity to the shape and the interior

To first order (Lecture 2), Clairaut's theorem relates $\beta$ to the flattening and the rotation parameter $m$:

$$\beta = \tfrac52 m - f \qquad\Longrightarrow\qquad f = \tfrac52 m - \beta . \tag{2}$$

The second-order version (Heiskanen & Moritz 1967, Eq. 2-115), which you may simply use, is

$$\beta = \tfrac52 m - f - \tfrac{17}{14}\,m f + \tfrac{15}{4}\,m^2 \qquad\Longrightarrow\qquad f = \frac{\tfrac52 m + \tfrac{15}{4}m^2 - \beta}{1 + \tfrac{17}{14}m}. \tag{3}$$

Once you have $f$, the shape relation $f \approx \tfrac32 J_2 + \tfrac12 m$ (Lecture 2, slide 16) gives

$$J_2 \approx \tfrac23\left(f - \tfrac12 m\right), \tag{4}$$

and the precession constant $H = (C - A)/C$ gives the moment of inertia, $C/Ma^2 = J_2/H$ (slide 17).

## <span style="color:green">Question 4</span>


(a) Compute $f$ from your $\beta$ with the first-order (Eq. 2) and the second-order (Eq. 3) formula, and express both as $1/f$. Which is closer to the GRS80 value 1/298.257? How large is the first-order error, and why is it so much larger than the 0.1 % error of $f \approx \tfrac32 J_2 + \tfrac12 m$ that we saw in lecture? (Hint: compare the size of $\beta$ and $f$ with the size of the neglected terms.)

(b) Propagate the formal uncertainty of $\beta$ into $1/f$.

(c) Compute $J_2$ (Eq. 4) and $C/Ma^2$, and compare with the satellite value $J_2 = 1.0826\times10^{-3}$ and with $C/Ma^2 = 0.3307$. What does your $C/Ma^2$ say about the Earth's interior?

## Part 5: Why Lapland and Peru?

Until the nineteenth century almost all gravity and arc measurements were made in western Europe. Let us see what that does to the result.

## <span style="color:green">Question 5 (graduate students only)</span>


(a) Select only the stations between 35°N and 70°N and between 10°W and 40°E ("Europe"). How many are there? Repeat the fit of Question 3 and the second-order flattening of Question 4 for this subset, with its formal uncertainty (Hint: The code for this remains under control, if you move your code from Question 4 into a function `fit_normal_gravity(lat_deg, g)` that takes station latitude in degrees `lat_deg` and the gravity observations `g` as vectors and returns `gamma_e`, `beta`, `s_gamma_e`, `s_beta`, and `r` the residual vector, which are then evaluated outside of the function.)

(b) Now select all stations between 40°N and 70°N, at any longitude. Fit again (see, that function would be really handy now).

(c) Compare the three results (global, Europe, 40--70°N) with each other and with GRS80. Explain, using the form of Eq. 1, why a narrow range of latitudes gives a poor estimate of $\beta$ even when there are many stations, and why the French Academy sent one expedition near the Arctic Circle and one to the equator. Which of the two problems, the narrow latitude range or the regional anomalies, dominates for the Europe subset?

## Part 6: What is left over

## <span style="color:green">Question 6</span>


Make a map of the residuals of the global fit (Question 3): plot the stations at their longitude and latitude, colored by the residual, with a diverging color map centered on zero (e.g. `cmap='RdBu_r'` with `vmin=-100, vmax=100`). Compare with the geoid map in Lecture 2 (slide 24). Where are the largest positive and negative anomalies? Do they look like the topography? Name one region where the anomalies are clearly related to plate tectonics.